In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp04/ex_2.py ──
"""
Shared infrastructure for MLFP04 Exercise 2 — EM and Gaussian Mixture Models.

Contains: synthetic GMM data generation, Singapore e-commerce loader, scaler,
scoring helpers, and OUTPUT_DIR management.

Technique-specific code (the EM E-step / M-step from scratch, BIC/AIC sweeps,
covariance-type comparison, mixture-of-experts gating) does NOT live here.
It lives in the per-technique files under modules/mlfp04/solutions/ex_2/.
"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex2_gmm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 2D DATA — 3 well-separated Gaussians
# ════════════════════════════════════════════════════════════════════════
#
# Used to validate the from-scratch EM implementation. Three well-separated
# components make convergence obvious and let students verify the recovered
# parameters against the ground truth.

TRUE_MEANS: np.ndarray = np.array([[0.0, 0.0], [5.0, 2.0], [2.0, 6.0]])
TRUE_COVS: np.ndarray = np.array(
    [
        [[1.0, 0.3], [0.3, 0.8]],
        [[0.8, -0.2], [-0.2, 1.2]],
        [[1.5, 0.0], [0.0, 0.5]],
    ]
)
TRUE_WEIGHTS: np.ndarray = np.array([0.4, 0.35, 0.25])
N_SYNTH: int = 600


def make_synthetic_gmm(
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw N_SYNTH points from the 3-component GMM defined by TRUE_*.

    Returns (X, z_true) where z_true is the ground-truth component index.
    """
    rng = np.random.default_rng(seed)
    n_per = (TRUE_WEIGHTS * N_SYNTH).astype(int)
    n_per[-1] = N_SYNTH - n_per[:-1].sum()

    parts: list[np.ndarray] = []
    labels: list[int] = []
    for k, (mean, cov, n) in enumerate(zip(TRUE_MEANS, TRUE_COVS, n_per)):
        parts.append(rng.multivariate_normal(mean, cov, n))
        labels.extend([k] * n)

    X = np.vstack(parts)
    z = np.array(labels)

    # Shuffle so the order does not leak the label
    idx = rng.permutation(N_SYNTH)
    return X[idx], z[idx]


# ════════════════════════════════════════════════════════════════════════
# REAL DATA — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════
#
# We reuse the MLFP03 e-commerce customer dataset (~6K rows, Singapore)
# for every real-data task in this exercise. Segmentation is the business
# frame: the GMM will propose soft customer segments that marketing can
# score on expected value.


def load_customers_scaled() -> (
    tuple[np.ndarray, pl.DataFrame, list[str], StandardScaler]
):
    """Return (X_scaled, customers_df, feature_cols, scaler).

    The DataFrame and feature_cols are returned so technique files can
    join cluster labels back onto the original rows for segment profiling.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    customers = customers.drop_nulls(subset=feature_cols)
    X, _, _ = to_sklearn_input(customers, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, customers, feature_cols, scaler


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING — for BIC/AIC interpretation
# ════════════════════════════════════════════════════════════════════════


def count_gmm_params(n_components: int, n_features: int, cov_type: str) -> int:
    """Number of free parameters in a GMM given components, features, cov type.

    Used to explain the BIC/AIC ranking across covariance types.
    """
    d = n_features
    k = n_components
    if cov_type == "full":
        return k * (d * (d + 1) // 2 + d + 1) - 1
    if cov_type == "tied":
        return d * (d + 1) // 2 + k * (d + 1) - 1
    if cov_type == "diag":
        return k * (2 * d + 1) - 1
    if cov_type == "spherical":
        return k * (d + 2) - 1
    raise ValueError(f"Unknown cov_type: {cov_type}")


# ════════════════════════════════════════════════════════════════════════
# SCORING HELPERS
# ════════════════════════════════════════════════════════════════════════


def safe_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    """Silhouette with a graceful fallback when only one cluster is present."""
    if len(set(labels.tolist())) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 2.3: Covariance Types — Shape vs Parsimony
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Distinguish full / tied / diag / spherical covariance types
#   - Count parameters for each type and see the complexity jump
#   - Run the same BIC sweep across all four types and pick a winner
#   - Explain why spherical GMM is just soft K-means
#   - Recognise when a simpler cov type is a better business choice
#
# PREREQUISITES: 02_sklearn_gmm.py
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — four cluster shapes, four parameter counts
#   2. Build — compare_cov_types helper
#   3. Train — fit all four cov types at the BIC-optimal K
#   4. Visualise — BIC per covariance type
#   5. Apply — Grab fraud-pattern segmentation (Singapore)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.mixture import GaussianMixture

from kailash_ml import ModelVisualizer



## THEORY — Four cluster shapes


In [ ]:
#   full:      each component has its own full covariance (most flexible)
#   tied:      all components share one covariance (same shape, diff centre)
#   diag:      each component has a diagonal covariance (axis-aligned)
#   spherical: each component has a scalar variance (equivalent to K-means)



## TASK 2 — BUILD: compare_cov_types


Fit a GMM of size k under each cov_type and return metric dicts.


In [ ]:
def compare_cov_types(
    X: np.ndarray,
    k: int,
    cov_types: tuple[str, ...] = ("full", "tied", "diag", "spherical"),
) -> dict[str, dict[str, float]]:
    n_features = X.shape[1]
    results: dict[str, dict[str, float]] = {}
    for ct in cov_types:
        # TODO: construct GaussianMixture with covariance_type=ct, n_components=k,
        # random_state=42, max_iter=200 — then .fit(X)
        gmm = ____
        ____
        labels = gmm.predict(X)
        # TODO: fill metrics — gmm.bic(X), gmm.aic(X), gmm.score(X)*n,
        # safe_silhouette(X, labels), count_gmm_params(k, n_features, ct)
        results[ct] = {
            "bic": ____,
            "aic": ____,
            "log_lik": ____,
            "silhouette": ____,
            "n_params": ____,
        }
    return results



## TASK 3 — TRAIN: fit each cov type at the BIC-optimal K


In [ ]:
X_scaled, _, feature_cols, _ = load_customers_scaled()
n_features = X_scaled.shape[1]

# Re-derive BIC-optimal K quickly
k_range = range(2, 9)
best_k, best_bic = None, float("inf")
for k in k_range:
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42).fit(
        X_scaled
    )
    b = float(gmm.bic(X_scaled))
    if b < best_bic:
        best_bic, best_k = b, k

print("=" * 70)
print(f"  Covariance comparison at BIC-optimal K={best_k}")
print("=" * 70)
print(f"Features: {n_features}  Rows: {X_scaled.shape[0]}")

# TODO: call compare_cov_types(X_scaled, best_k)
cov_results = ____

print(
    f"\n{'cov_type':<12} {'BIC':>12} {'log_lik':>12} "
    f"{'silhouette':>12} {'params':>8}"
)
print("─" * 60)
for ct, v in cov_results.items():
    print(
        f"{ct:<12} {v['bic']:>12.0f} {v['log_lik']:>12.0f} "
        f"{v['silhouette']:>12.4f} {int(v['n_params']):>8}"
    )

# TODO: pick the cov type with the lowest BIC
best_cov = ____
print(f"\nBest covariance type by BIC: {best_cov}")



### Checkpoint 1


In [ ]:
assert len(cov_results) == 4, "should have 4 covariance types"
assert (
    cov_results["full"]["n_params"] > cov_results["spherical"]["n_params"]
), "full covariance must have more parameters than spherical"
assert best_cov in cov_results, "best cov must be one of the fitted types"
print("\n[ok] Checkpoint 1 passed — all four cov types fitted and ranked by BIC")



## TASK 4 — VISUALISE: BIC per covariance type


In [ ]:
viz = ModelVisualizer()
comparison = {
    f"cov={ct}": {"BIC": v["bic"], "silhouette": v["silhouette"]}
    for ct, v in cov_results.items()
}
# TODO: viz.metric_comparison(comparison)
fig = ____
fig.update_layout(title=f"Covariance shape vs BIC (K={best_k})")
fig.write_html(str(out_path("ex2_covariance_comparison.html")))
print(f"\nSaved: {out_path('ex2_covariance_comparison.html')}")



### Checkpoint 2


In [ ]:
assert out_path("ex2_covariance_comparison.html").exists(), "plot must exist"
print("[ok] Checkpoint 2 passed — covariance comparison chart written")



## TASK 5 — APPLY: Grab fraud-pattern segmentation (Singapore)


In [ ]:
# SCENARIO: Grab processes ~6M daily transactions across SEA. The risk
# team must separate normal activity from fraud patterns — stolen-card
# testing, promo abuse, driver collusion — WITHOUT labels.
#
# Why covariance type matters:
#   Normal transactions cluster as tight, axis-aligned blobs (diagonal
#   cov fits cheaply). Fraud patterns are CORRELATED: e.g. "small
#   amount + odd hour + new card" moves together. Capturing the
#   rotation needs full covariance.
#
# BUSINESS IMPACT (from Grab risk disclosures):
#   - ~24,000 daily fraud attempts
#   - Current rules-based recall: ~70% => ~S$612K/day = S$223M/year losses
#   - Full-cov GMM on fraud clusters lifts recall ~6 points
#   - => ~S$44.7M/year in avoided losses vs <S$20K/year extra compute
#   - Return: >2,000x.

print("\n" + "=" * 70)
print("  APPLY — Grab fraud pattern segmentation")
print("=" * 70)
print(
    f"At BIC-optimal K={best_k}, covariance winner: {best_cov}. "
    "For fraud workloads the risk team usually splits the population: "
    "'diag' on the bulk of normal traffic and 'full' on the suspicious tail."
)

print("\nParameter count per covariance type at the same K:")
for ct, v in cov_results.items():
    print(f"  {ct:<12} -> {int(v['n_params']):>6} parameters")



## REFLECTION


[x] Four cov shapes: full > tied > diag > spherical
  [x] Parameter count scales with d^2 (full) down to a scalar (spherical)
  [x] BIC automatically trades flexibility against parsimony
  [x] Grab fraud scenario: full-cov unlocks ~S$44.7M/year in blocked losses

  Next: 04_mixture_of_experts.py — soft vs hard assignments, and the
  bridge from GMMs to modern LLM routing.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

